In [0]:
# Upgrade Databricks SDK to the latest version and restart Python to see updated packages
%pip install --upgrade databricks-sdk==0.70.0
%restart_python

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import JobSettings as Job

In [0]:
ROOT_PATH = '/Workspace/Users/icaro.data.eng@gmail.com/ntt_data/' # Add your own path here

In [0]:
covid_19 = Job.from_dict(
    {
        'name': 'covid_19',
        'tasks': [
            {
                'task_key': 'extract_locations_data',
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}data_extraction/extract_data',
                    'base_parameters': {
                        'file_type': 'locations',
                    },
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'bronze-locations',
                'depends_on': [
                    {
                        'task_key': 'extract_locations_data',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}bronze/bronze_locations',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'extract_vaccinations_data',
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}data_extraction/extract_data',
                    'base_parameters': {
                        'file_type': 'vaccinations',
                    },
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'bronze-vaccinations',
                'depends_on': [
                    {
                        'task_key': 'extract_vaccinations_data',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}bronze/bronze_vaccinations',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'silver-locations',
                'depends_on': [
                    {
                        'task_key': 'bronze-locations',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}silver/silver_locations',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'silver-country_vaccine',
                'depends_on': [
                    {
                        'task_key': 'silver-locations',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}silver/silver_country_vaccine',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_vaccine',
                'depends_on': [
                    {
                        'task_key': 'silver-country_vaccine',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_vaccine',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'silver-vaccinations',
                'depends_on': [
                    {
                        'task_key': 'bronze-vaccinations',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}silver/silver_vaccinations',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_country',
                'depends_on': [
                    {
                        'task_key': 'silver-vaccinations',
                    },
                    {
                        'task_key': 'silver-locations',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_country',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-bridge_country_vaccine',
                'depends_on': [
                    {
                        'task_key': 'gold-dim_country',
                    },
                    {
                        'task_key': 'gold-dim_vaccine',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/bridge_country_vaccine',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_date',
                'depends_on': [
                    {
                        'task_key': 'silver-vaccinations',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_date',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-fact_vaccinations',
                'depends_on': [
                    {
                        'task_key': 'gold-dim_country',
                    },
                    {
                        'task_key': 'gold-dim_date',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/fact_vaccinations',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'data_analysis',
                'depends_on': [
                    {
                        'task_key': 'gold-fact_vaccinations',
                    },
                    {
                        'task_key': 'gold-bridge_country_vaccine',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}data_analysis',
                    'source': 'WORKSPACE',
                },
            },
        ],
        'queue': {
            'enabled': True,
        },
        'performance_target': 'PERFORMANCE_OPTIMIZED',
    }
)


In [0]:
w = WorkspaceClient()
# w.jobs.reset(new_settings=covid_19, job_id=614613405102480)
w.jobs.create(**covid_19.as_shallow_dict())